Importing Basic Libraries

In [ ]:
import pandas as pd
import numpy as np

Importing Regression Funcitons

In [ ]:
from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
import lightgbm as lgb

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

Importing Evaluator Functions

In [ ]:
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 9.8 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import r2_score

from sklearn.metrics import mean_absolute_percentage_error
from sktime.performance_metrics.forecasting import median_absolute_percentage_error

Importing Classes and Functions from Alternate Files

In [ ]:
from model_classes import W4_Regression, W5_Regs, W7_Boosting

In [ ]:
# Preset file

filename = "CRMLS_0625-0626_enriched.csv"
end_mnth = 6

In [ ]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [ ]:
enr_df = load_df()

In [ ]:
print(f"Column Check: \t {enr_df.shape}")

Column Check: 	 (102071, 64)


In [ ]:
# Preset Values

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols


cols = enr_df.columns.to_list()
new_col_strt = [i for i in range(len(cols)) if cols[i] == "SaleMonth"]
new_cols = cols[(new_col_strt[0]+1):len(cols)]
new_cols = [i for i in new_cols if i != "DistrictName"]

crit_cols = totals+new_cols
log_cols = main_cols+new_cols


target = "ClosePrice"

In [ ]:
def save_csv(df, file):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(file, index=False)

In [ ]:
class W8_Eval():

  def __init__(self,df):
    self.df = df


  def mape_eval(self, y_test, y_pred):
    mape = mean_absolute_percentage_error(y_test, y_pred)       # Computes the mean absolute percentage error of y_test and the predicted y
    return mape


  def mdape_eval(self, y_test, y_pred):
    mdape = median_absolute_percentage_error(y_test, y_pred)       # Computes the median absolute percentage error of y_test and the predicted y
    return mdape

In [ ]:
base = W4_Regression(enr_df)
comp = W5_Regs(enr_df)
boost = W7_Boosting(enr_df)
Week8 = W8_Eval(enr_df)

train, test = base.test_train_split()


log_df = boost.log_transform()

bs = W4_Regression(log_df)
cp = W5_Regs(log_df)
bst = W7_Boosting(log_df)
Wk8 = W8_Eval(log_df)

tr, te = bs.test_train_split()

# **Evaluation:  *MAPE, MdAPE***

 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

**Linear Regression**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LRmape, LRmape = comp.shrt_main(base.LinReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmape, logLRmape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LRmdape, LRmdape = comp.shrt_main(base.LinReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmdape, logLRmdape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
Correlation of ['AvgAreaCost']:
 0.2703
Log Transform
Correlation of ['AvgAreaCost', 'LivingArea', 'BathroomsTotalInteger', 'AvgLivingArea', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'BathroomsTotalInteger/BedroomsTotal', 'AvgAreaLot', 'LivingArea/LotSizeSquareFeet', 'DistrictID', 'PoolPrivateYN_True', 'PostalCode', 'LotSizeSquareFeet/DaysOnMarket', 'LotSizeSquareFeet', 'NewConstructionYN_True', 'DaysOnMarket', 'YearBuilt', 'ViewYN_False', 'ViewYN_True', 'NewConstructionYN_False']:
 0.0108


Median Absolute Percentage Error 
Non-Transform
Correlation of ['AvgAreaCost']:
 0.18
Log Transform
Correlation of ['AvgAreaCost', 'LivingArea', 'FireplaceYN_False', 'FireplaceYN_True', 'AvgLivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea/DaysOnMarket', 'AvgAreaLot', 'PoolPrivateYN_False', 'BathroomsTotalInteger/BedroomsTotal', 'NewConstructionYN_True', 'DistrictID', 'Liv

**Decision Tree Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_DTRmape, DTRmape = comp.shrt_main(comp.TreeReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmape, logDTRmape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_DTRmdape, DTRmdape = comp.shrt_main(comp.TreeReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmdape, logDTRmdape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'PoolPrivateYN_True', 'YearBuilt', 'NewConstructionYN_False', 'NewConstructionYN_True']:
 0.1754
Log Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LotSizeSquareFeet', 'YearBuilt']:
 0.0124


Median Absolute Percentage Error 
Non-Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgAreaLot', 'AvgLivingArea', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'LotSizeSquareFeet', 'BedroomsTotal', 'LotSizeSquareFeet/DaysOnM

**Random Forest Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_RFRmape, RFRmape = comp.shrt_main(comp.ForestReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmape, logRFRmape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'PoolPrivateYN_True', 'YearBuilt', 'NewConstructionYN_False', 'NewConstructionYN_True', 'LivingArea/DaysOnMarket', 'ViewYN_False', 'ViewYN_True', 'DaysOnMarket']:
 0.129
Log Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'YearBuilt', 'PoolPrivateYN_True', 'NewConstructionYN_True', 'DaysOnMarket', 'ViewYN_False', 'ViewYN_True', 'NewConstructionYN_False', 'SaleMonth', 'LotSizeSquareFeet/DaysOnMarket']:
 0.0089


In [ ]:
# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_RFRmdape, RFRmdape = comp.shrt_main(comp.ForestReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmdape, logRFRmdape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Median Absolute Percentage Error 
Non-Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet/DaysOnMarket', 'LivingArea/LotSizeSquareFeet', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'YearBuilt', 'ViewYN_True', 'ViewYN_False', 'PoolPrivateYN_True', 'DaysOnMarket']:
 0.0822
Log Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'LotSizeSquareFeet', 'BathroomsTotalInteger', 'FireplaceYN_False', 'FireplaceYN_True', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'YearBuilt', 'PoolPrivateYN_False', 'NewConstructionYN_True', 'DaysOnMarket', 'PoolPrivateYN_True', 'ViewYN_False', 'ViewYN_True']:
 0.0061


In [ ]:
def load_data(file):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  return df

In [ ]:
bst_r2s = load_data("boosting_r2s.csv")

**XGBoost**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_XGBmape, XGBmape = comp.shrt_main(boost.XGB, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                    dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])

print("Log Transform")
c_logXGBmape, logXGBmape = cp.shrt_main(bst.XGB, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_XGBmdape, XGBmdape = comp.shrt_main(boost.XGB, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])

print("Log Transform")
c_logXGBmdape, logXGBmdape = cp.shrt_main(bst.XGB, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])

Mean Absolute Percentage Error
Non-Transform
Correlation of ['AvgAreaCost', 'DistrictID', 'PostalCode', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_True', 'YearBuilt', 'LotSizeSquareFeet/DaysOnMarket', 'NewConstructionYN_False', 'NewConstructionYN_True', 'ViewYN_False', 'ViewYN_True', 'DaysOnMarket']:
 0.1404
Log Transform
Correlation of ['AvgAreaCost', 'DistrictID', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'YearBuilt', 'LotSizeSquareFeet/DaysOnMarket', 'PoolPrivateYN_True', 'DaysOnMarket', 'NewConstructionYN_T

**Gradient Boosting Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_GBRmape, GBRmape = comp.shrt_main(boost.GBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                    dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])

print("Log Transform")
c_logGBRmape, logGBRmape = cp.shrt_main(bst.GBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
# print("Non-Transform")
c_GBRmdape, GBRmdape = comp.shrt_main(boost.GBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])

print("Log Transform")
c_logGBRmdape, logGBRmdape = cp.shrt_main(bst.GBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])

Mean Absolute Percentage Error
Non-Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_True', 'LotSizeSquareFeet/DaysOnMarket', 'YearBuilt', 'NewConstructionYN_False', 'NewConstructionYN_True', 'ViewYN_False', 'DaysOnMarket']:
 0.1419
Log Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'LotSizeSquareFeet', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LotSizeSquareFeet/DaysOnMarket', 'YearBuilt', 'PoolPrivateYN_True', 'DaysOnMarket', 'NewConstructionYN_True', 'ViewYN_F

**LightGBM**

In [ ]:
# MAPE

In [ ]:
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LGBMmape, LGBMmape = comp.shrt_main(boost.L_GBM, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])

Mean Absolute Percentage Error
Non-Transform
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001876 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 1254042.147315
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

In [ ]:
print("Log Transform")
c_logLGBMmape, logLGBMmape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])

Log Transform
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=6) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=64) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=6) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=64) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001417 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 13.785278
[LightGBM] [Warning] No further splits with 

In [ ]:
# MdAPE

In [ ]:
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LGBMmdape, LGBMmdape = comp.shrt_main(boost.L_GBM, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])

Median Absolute Percentage Error 
Non-Transform
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002923 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 1254042.147315
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warnin

In [ ]:
print("Log Transform")
c_logLGBMmdape, logLGBMmdape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                            dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])

Log Transform
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=6) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=64) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=6) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=64) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001332 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 13.785278
[LightGBM] [Warning] No further splits with 

**LightGBM Summary**

***Mean Absolute Percentage Error***

*Non-Transform*

* ***0.162***

*Log Transform*

* ***0.0091***

***Median Absolute Percentage Error***

*Non-Transform*

* ***0.1096***

*Log Transform*

* ***0.0064***

**Histogram-based Gradient Boosting Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_HGBRmape, HGBRmape = comp.shrt_main(boost.HGBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])

print("Log Transform")
c_logHGBRmape, logHGBRmape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_HGBRmdape, HGBRmdape = comp.shrt_main(boost.HGBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])

print("Log Transform")
c_logHGBRmdape, logHGBRmdape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                            dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])

Mean Absolute Percentage Error
Non-Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'LotSizeSquareFeet', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_True', 'YearBuilt']:
 0.1454
Log Transform
Correlation of ['AvgAreaCost', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'YearBuilt', 'PoolPrivateYN_True', 'LotSizeSquareFeet/DaysOnMarket', 'DaysOnMarket', 'NewConstructionYN_True', 'ViewYN_False', 'ViewYN_True']:
 0.0091


Median Absolute Percentage Error 
Non-Transform
Correlation of ['AvgAreaCost', 'Postal

***Summary***

---

 Lowest MAPE:
   * Non-Transform: **0.129**
     * **Random Forest**
   * Log Transform: **0.0086**
     * **XGBoost**

---

 Lowest MdAPE:
   * Non-Transform: **0.0822**
     * **Random Forest**
   * Log Transform: **0.0059**
     * **XGBoost**

# **Price Bands**

In [ ]:
# Defines column "PriceRank" with .qcut from 'Low' to 'High'
enr_df["PriceRank"] = pd.qcut(enr_df["ClosePrice"], q=5, labels=["Low", "Low-Medium", "Medium", "Medium-High", "High"])

**Bin Values**
 * ***Q1: High***
   * `$1,610,000 - $110,000,000`
   * (1610000.0, 110000000.0]
 * ***Q2: Medium-High***
   * `$1,086,000 - $1,610,000`
   * (1086000.0, 1610000.0]
 * ***Q3: Medium***
   * `$791,000 - $1,086,000`
   * (791000.0, 1086000.0]
 * ***Q4: Low-Medium***
   * `$574,000 - $791,000`
   * (574000.0, 791000.0]
 * ***Q5: Low***
   * `$26,000 - $574,000`
   * (25999.999, 574000.0]

In [ ]:
# Bin Values
(sorted(pd.unique(pd.qcut(enr_df["ClosePrice"], q=5)), reverse=True))

[Interval(1610000.0, 110000000.0, closed='right'),
 Interval(1086000.0, 1610000.0, closed='right'),
 Interval(791000.0, 1086000.0, closed='right'),
 Interval(574000.0, 791000.0, closed='right'),
 Interval(25999.999, 574000.0, closed='right')]

In [ ]:
def quant_eval(df, q_val, modl_typ, feats=crit_cols):

  qdf = df[df["PriceRank"] == q_val]
  qdf.reset_index(drop=True, inplace=True)
  ct = len(qdf)

  wk4, wk5 = W4_Regression(qdf), W5_Regs(qdf)
  wk7, wk8 = W7_Boosting(qdf), W8_Eval(qdf)
  train, test = wk4.test_train_split()

  qlog_df = wk7.log_transform()
  w4, w5 = W4_Regression(qlog_df), W5_Regs(qlog_df)
  w7, w8 = W7_Boosting(qlog_df), W8_Eval(qlog_df)
  tr, te = w4.test_train_split()


  if modl_typ == "Linear Regression":              # Linear Regression
    r2 = wk5.shrt_main(wk4.LinReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk4.LinReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk4.LinReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w4.LinReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w4.LinReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w4.LinReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Decision Tree Regressor":             # Decision Tree Regressor
    r2 = wk5.shrt_main(wk5.TreeReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk5.TreeReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk5.TreeReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w5.TreeReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w5.TreeReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w5.TreeReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Random Forest Regressor":             # Random Forest Regressor
    r2 = wk5.shrt_main(wk5.ForestReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk5.ForestReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk5.ForestReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w5.ForestReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w5.ForestReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w5.ForestReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "XGBoost":             # XGBoost

    r2 = wk5.shrt_main(wk7.XGB, train, test, wk4.r2_eval, crit_cols, prt=False,)
    mape = wk5.shrt_main(wk7.XGB, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])
    mdape = wk5.shrt_main(wk7.XGB, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])

    lg_r2 = w5.shrt_main(w7.XGB, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.XGB, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])
    lg_mdape = w5.shrt_main(w7.XGB, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])


  if modl_typ == "Gradient Boosting Regressor":             # Gradient Boosting Regressor

    r2 = wk5.shrt_main(wk7.GBR, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.GBR, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])
    mdape = wk5.shrt_main(wk7.GBR, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])

    lg_r2 = w5.shrt_main(w7.GBR, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.GBR, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])
    lg_mdape = w5.shrt_main(w7.GBR, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])


  if modl_typ == "LightGBM":             # LightGBM

    r2 = wk5.shrt_main(wk7.L_GBM, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.L_GBM, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])
    mdape = wk5.shrt_main(wk7.L_GBM, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])

    lg_r2 = w5.shrt_main(w7.L_GBM, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.L_GBM, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])
    lg_mdape = w5.shrt_main(w7.L_GBM, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])


  if modl_typ == "Histogram-based Gradient Boosting Regressor":             # Histogram-based Gradient Boosting Regressor

    r2 = wk5.shrt_main(wk7.HGBR, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.HGBR, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])
    mdape = wk5.shrt_main(wk7.HGBR, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])

    lg_r2 = w5.shrt_main(w7.HGBR, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.HGBR, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])
    lg_mdape = w5.shrt_main(w7.HGBR, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])

  new_df = pd.DataFrame({"Model": [modl_typ]*2, "PriceRank": q_val, "RowCount": ct, "LogForm": [False,True], "R2 Score": [r2[1],lg_r2[1]], "MAPE": [mape[1],lg_mape[1]], "MdAPE": [mdape[1],lg_mdape[1]]})
  return new_df

In [ ]:
qdfLR1 = quant_eval(enr_df, "Low", "Linear Regression")

In [ ]:
qdfLR2 = quant_eval(enr_df, "Low-Medium", "Linear Regression")

In [ ]:
qdfLR3 = quant_eval(enr_df, "Medium", "Linear Regression")

In [ ]:
qdfLR4 = quant_eval(enr_df, "Medium-High", "Linear Regression")

In [ ]:
qdfLR5 = quant_eval(enr_df, "High", "Linear Regression")

In [ ]:
qdfLR = pd.concat([qdfLR1, qdfLR2, qdfLR3, qdfLR4, qdfLR5])
qdfLR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfLR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Linear Regression,Low,20417,False,0.448506,0.180019,0.108916
1,Linear Regression,Low-Medium,20416,False,0.167935,0.074288,0.072855
2,Linear Regression,Medium,20412,False,0.170838,0.064090,0.058561
3,Linear Regression,Medium-High,20499,False,0.221882,0.082979,0.076220
4,Linear Regression,High,20327,False,0.649954,0.249541,0.192221
5,Linear Regression,Low,20417,True,0.576078,0.011029,0.007839
6,Linear Regression,Low-Medium,20416,True,0.246339,0.005169,0.004850
7,Linear Regression,Medium,20412,True,0.240385,0.004478,0.003960
8,Linear Regression,Medium-High,20499,True,0.290935,0.005473,0.004901
9,Linear Regression,High,20327,True,0.733996,0.011312,0.008241


In [ ]:
qdfDTR1 = quant_eval(enr_df, "Low", "Decision Tree Regressor")

In [ ]:
qdfDTR2 = quant_eval(enr_df, "Low-Medium", "Decision Tree Regressor")

In [ ]:
qdfDTR3 = quant_eval(enr_df, "Medium", "Decision Tree Regressor")

In [ ]:
qdfDTR4 = quant_eval(enr_df, "Medium-High", "Decision Tree Regressor")

In [ ]:
qdfDTR5 = quant_eval(enr_df, "High", "Decision Tree Regressor")

In [ ]:
qdfDTR = pd.concat([qdfDTR1, qdfDTR2, qdfDTR3, qdfDTR4, qdfDTR5])
qdfDTR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfDTR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Decision Tree Regressor,Low,20417,False,0.569465,0.153641,0.083728
1,Decision Tree Regressor,Low-Medium,20416,False,0.197368,0.070971,0.052198
2,Decision Tree Regressor,Medium,20412,False,0.137129,0.065420,0.060074
3,Decision Tree Regressor,Medium-High,20499,False,0.111330,0.084870,0.076233
4,Decision Tree Regressor,High,20327,False,0.563848,0.190982,0.128205
5,Decision Tree Regressor,Low,20417,True,0.571398,0.010667,0.006837
6,Decision Tree Regressor,Low-Medium,20416,True,0.197233,0.005270,0.003876
7,Decision Tree Regressor,Medium,20412,True,0.138785,0.004778,0.004266
8,Decision Tree Regressor,Medium-High,20499,True,0.119472,0.005988,0.005457
9,Decision Tree Regressor,High,20327,True,0.681839,0.012355,0.008557


In [ ]:
qdfRFR1 = quant_eval(enr_df, "Low", "Random Forest Regressor")

In [ ]:
qdfRFR2 = quant_eval(enr_df, "Low-Medium", "Random Forest Regressor")

In [ ]:
qdfRFR3 = quant_eval(enr_df, "Medium", "Random Forest Regressor")

In [ ]:
qdfRFR4 = quant_eval(enr_df, "Medium-High", "Random Forest Regressor")

In [ ]:
qdfRFR5 = quant_eval(enr_df, "High", "Random Forest Regressor")

In [ ]:
qdfRFR = pd.concat([qdfRFR1, qdfRFR2, qdfRFR3, qdfRFR4, qdfRFR5])
qdfRFR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfRFR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Random Forest Regressor,Low,20417,False,0.719019,0.117334,0.066031
1,Random Forest Regressor,Low-Medium,20416,False,0.486095,0.054326,0.045086
2,Random Forest Regressor,Medium,20412,False,0.347920,0.055795,0.047373
3,Random Forest Regressor,Medium-High,20499,False,0.373899,0.071975,0.061396
4,Random Forest Regressor,High,20327,False,0.735261,0.149119,0.101955
5,Random Forest Regressor,Low,20417,True,0.699618,0.008512,0.005199
6,Random Forest Regressor,Low-Medium,20416,True,0.484617,0.004056,0.003385
7,Random Forest Regressor,Medium,20412,True,0.352609,0.004066,0.003452
8,Random Forest Regressor,Medium-High,20499,True,0.381259,0.005018,0.004272
9,Random Forest Regressor,High,20327,True,0.796726,0.009630,0.006680


In [ ]:
qdfXGB1 = quant_eval(enr_df, "Low", "XGBoost")

In [ ]:
qdfXGB2 = quant_eval(enr_df, "Low-Medium", "XGBoost")

In [ ]:
qdfXGB3 = quant_eval(enr_df, "Medium", "XGBoost")

In [ ]:
qdfXGB4 = quant_eval(enr_df, "Medium-High", "XGBoost")

In [ ]:
qdfXGB5 = quant_eval(enr_df, "High", "XGBoost")

In [ ]:
qdfXGB = pd.concat([qdfXGB1, qdfXGB2, qdfXGB3, qdfXGB4, qdfXGB5])
qdfXGB.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfXGB

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,XGBoost,Low,20417,False,0.722196,0.118333,0.068325
1,XGBoost,Low-Medium,20416,False,0.482016,0.055810,0.045188
2,XGBoost,Medium,20412,False,0.334256,0.058227,0.045773
3,XGBoost,Medium-High,20499,False,0.350439,0.074858,0.059111
4,XGBoost,High,20327,False,0.748880,0.157622,0.108219
5,XGBoost,Low,20417,True,0.696924,0.008278,0.004885
6,XGBoost,Low-Medium,20416,True,0.482540,0.003965,0.003242
7,XGBoost,Medium,20412,True,0.323753,0.003986,0.003290
8,XGBoost,Medium-High,20499,True,0.356604,0.004954,0.004123
9,XGBoost,High,20327,True,0.803197,0.009360,0.006483


In [ ]:
qdfGBR1 = quant_eval(enr_df, "Low", "Gradient Boosting Regressor")

In [ ]:
qdfGBR2 = quant_eval(enr_df, "Low-Medium", "Gradient Boosting Regressor")

In [ ]:
qdfGBR3 = quant_eval(enr_df, "Medium", "Gradient Boosting Regressor")

In [ ]:
qdfGBR4 = quant_eval(enr_df, "Medium-High", "Gradient Boosting Regressor")

In [ ]:
qdfGBR5 = quant_eval(enr_df, "High", "Gradient Boosting Regressor")

In [ ]:
qdfGBR = pd.concat([qdfGBR1, qdfGBR2, qdfGBR3, qdfGBR4, qdfGBR5])
qdfGBR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfGBR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Gradient Boosting Regressor,Low,20417,False,0.695711,0.115048,0.065850
1,Gradient Boosting Regressor,Low-Medium,20416,False,0.401496,0.054791,0.045480
2,Gradient Boosting Regressor,Medium,20412,False,0.318214,0.055220,0.048291
3,Gradient Boosting Regressor,Medium-High,20499,False,0.332102,0.071585,0.061387
4,Gradient Boosting Regressor,High,20327,False,0.712518,0.150130,0.106555
5,Gradient Boosting Regressor,Low,20417,True,0.692541,0.008547,0.005383
6,Gradient Boosting Regressor,Low-Medium,20416,True,0.405805,0.004049,0.003388
7,Gradient Boosting Regressor,Medium,20412,True,0.321434,0.004035,0.003444
8,Gradient Boosting Regressor,Medium-High,20499,True,0.332693,0.005039,0.004301
9,Gradient Boosting Regressor,High,20327,True,0.783422,0.009372,0.006757


In [ ]:
qdfLGBM1 = quant_eval(enr_df, "Low", "LightGBM")

In [ ]:
qdfLGBM2 = quant_eval(enr_df, "Low-Medium", "LightGBM")

In [ ]:
qdfLGBM3 = quant_eval(enr_df, "Medium", "LightGBM")

In [ ]:
qdfLGBM4 = quant_eval(enr_df, "Medium-High", "LightGBM")

In [ ]:
qdfLGBM5 = quant_eval(enr_df, "High", "LightGBM")

In [ ]:
qdfLGBM = pd.concat([qdfLGBM1, qdfLGBM2, qdfLGBM3, qdfLGBM4, qdfLGBM5])
qdfLGBM.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfLGBM

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,LightGBM,Low,20417,False,0.726896,0.118221,0.068002
1,LightGBM,Low-Medium,20416,False,0.480140,0.056928,0.048128
2,LightGBM,Medium,20412,False,0.363784,0.056021,0.048402
3,LightGBM,Medium-High,20499,False,0.377888,0.072781,0.063337
4,LightGBM,High,20327,False,0.740175,0.161357,0.115954
5,LightGBM,Low,20417,True,0.712329,0.008424,0.005310
6,LightGBM,Low-Medium,20416,True,0.481196,0.003993,0.003316
7,LightGBM,Medium,20412,True,0.367427,0.003997,0.003427
8,LightGBM,Medium-High,20499,True,0.384398,0.004981,0.004198
9,LightGBM,High,20327,True,0.811627,0.009345,0.006707


In [ ]:
qdfHGBR1 = quant_eval(enr_df, "Low", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR2 = quant_eval(enr_df, "Low-Medium", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR3 = quant_eval(enr_df, "Medium", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR4 = quant_eval(enr_df, "Medium-High", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR5 = quant_eval(enr_df, "High", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR = pd.concat([qdfHGBR1, qdfHGBR2, qdfHGBR3, qdfHGBR4, qdfHGBR5])
qdfHGBR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfHGBR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Histogram-based Gradient Boosting Regressor,Low,20417,False,0.723475,0.116061,0.065612
1,Histogram-based Gradient Boosting Regressor,Low-Medium,20416,False,0.476763,0.054102,0.044322
2,Histogram-based Gradient Boosting Regressor,Medium,20412,False,0.365551,0.055010,0.046195
3,Histogram-based Gradient Boosting Regressor,Medium-High,20499,False,0.383322,0.071368,0.060742
4,Histogram-based Gradient Boosting Regressor,High,20327,False,0.715875,0.156697,0.109519
5,Histogram-based Gradient Boosting Regressor,Low,20417,True,0.707449,0.008616,0.005277
6,Histogram-based Gradient Boosting Regressor,Low-Medium,20416,True,0.481987,0.004025,0.003276
7,Histogram-based Gradient Boosting Regressor,Medium,20412,True,0.364410,0.004005,0.003395
8,Histogram-based Gradient Boosting Regressor,Medium-High,20499,True,0.384351,0.004941,0.004202
9,Histogram-based Gradient Boosting Regressor,High,20327,True,0.809729,0.009357,0.006529


In [ ]:
qdf_all = pd.concat([qdfLR, qdfDTR, qdfRFR, qdfXGB, qdfGBR, qdfLGBM, qdfHGBR])
qdf_all

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Linear Regression,Low,20417,False,0.448506,0.180019,0.108916
1,Linear Regression,Low-Medium,20416,False,0.167935,0.074288,0.072855
2,Linear Regression,Medium,20412,False,0.170838,0.064090,0.058561
3,Linear Regression,Medium-High,20499,False,0.221882,0.082979,0.076220
4,Linear Regression,High,20327,False,0.649954,0.249541,0.192221
...,...,...,...,...,...,...,...
5,Histogram-based Gradient Boosting Regressor,Low,20417,True,0.707449,0.008616,0.005277
6,Histogram-based Gradient Boosting Regressor,Low-Medium,20416,True,0.481987,0.004025,0.003276
7,Histogram-based Gradient Boosting Regressor,Medium,20412,True,0.364410,0.004005,0.003395
8,Histogram-based Gradient Boosting Regressor,Medium-High,20499,True,0.384351,0.004941,0.004202


In [ ]:
save_csv(qdf_all, "price_band_metrics.csv")

# **Metrics DataFrame: *R2, MAPE, MdAPE***

 * R2 Score (R2)
 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

In [ ]:
reg_r2s = load_data("regression_r2s.csv")

In [ ]:
modl = ["Linear Regression"]*4 + ["Decision Tree Regressor"]*4 + ["Random Forest Regressor"]*4 + [
    "XGBoost"]*4 + ["Gradient Boosting Regressor"]*4 + ["LightGBM"]*4 + ["Histogram-based Gradient Boosting Regressor"]*4

lg = [False, True]*14

typ = ["MAPE", "MAPE", "MdAPE", "MdAPE"]*7

cols = [c_LRmape, c_logLRmape, c_LRmdape, c_logLRmdape] + [c_DTRmape, c_logDTRmape, c_DTRmdape, c_logDTRmdape] + [
    c_RFRmape, c_logRFRmape, c_RFRmdape, c_logRFRmdape] + [c_XGBmape, c_logXGBmape, c_XGBmdape, c_logXGBmdape] + [
        c_GBRmape, c_logGBRmape, c_GBRmdape, c_logGBRmdape] + [c_LGBMmape, c_logLGBMmape, c_LGBMmdape, c_logLGBMmdape] + [
        c_HGBRmape, c_logHGBRmape, c_HGBRmdape, c_logHGBRmdape]

scrs = [LRmape, logLRmape, LRmdape, logLRmdape] + [DTRmape, logDTRmape, DTRmdape, logDTRmdape] + [RFRmape, logRFRmape, RFRmdape, logRFRmdape] + [
    XGBmape, logXGBmape, XGBmdape, logXGBmdape] + [GBRmape, logGBRmape, GBRmdape, logGBRmdape] + [LGBMmape, logLGBMmape, LGBMmdape, logLGBMmdape] + [
        HGBRmape, logHGBRmape, HGBRmdape, logHGBRmdape]

In [ ]:
md = [None]*12 + [bst_r2s["max_depth"].iloc[i] for i in range(2)]*2 + [bst_r2s["max_depth"].iloc[i] for i in range(2,4)]*2 + [
    bst_r2s["max_depth"].iloc[i] for i in range(4,6)]*2 + [bst_r2s["max_depth"].iloc[i] for i in range(6,8)]*2

lr = [None]*12 + [bst_r2s["learning_rate"].iloc[i] for i in range(2)]*2 + [bst_r2s["learning_rate"].iloc[i] for i in range(2,4)]*2 + [
    bst_r2s["learning_rate"].iloc[i] for i in range(4,6)]*2 + [bst_r2s["learning_rate"].iloc[i] for i in range(6,8)]*2

ne = [None]*12 + [bst_r2s["n_estimators"].iloc[i] for i in range(2)]*2 + [bst_r2s["n_estimators"].iloc[i] for i in range(2,4)]*2 + [
    bst_r2s["n_estimators"].iloc[i] for i in range(4,6)]*2 + [bst_r2s["n_estimators"].iloc[i] for i in range(6,8)]*2

In [ ]:
mets = {"Model": modl, "LogForm": lg, "ScoreType": typ, "ScoreValue": scrs, "max_depth": md, "learning_rate": lr, "n_estimators": ne, "columns": cols}

met_df = pd.DataFrame(mets)
met_df = pd.concat([reg_r2s, bst_r2s, met_df])
met_df.sort_values(by=["Model", "ScoreType", "LogForm"], ignore_index=True, inplace=True)
met_df

,Model,LogForm,ScoreType,ScoreValue,max_depth,learning_rate,n_estimators,columns
0,Decision Tree Regressor,False,MAPE,0.175436,NaN,NaN,NaN,"[AvgAreaCost, PostalCode, AvgLivingArea, AvgAr..."
1,Decision Tree Regressor,True,MAPE,0.012368,NaN,NaN,NaN,"[AvgAreaCost, PostalCode, AvgLivingArea, AvgAr..."
2,Decision Tree Regressor,False,MdAPE,0.115385,NaN,NaN,NaN,"[AvgAreaCost, PostalCode, AvgAreaLot, AvgLivin..."
3,Decision Tree Regressor,True,MdAPE,0.008245,NaN,NaN,NaN,"[AvgAreaCost, PostalCode, AvgAreaLot, AvgLivin..."
4,Decision Tree Regressor,False,R2 Score,0.708986,NaN,NaN,NaN,"['AvgAreaCost', 'AvgLivingArea', 'AvgAreaLot',..."
5,Decision Tree Regressor,True,R2 Score,0.860948,NaN,NaN,NaN,"['AvgAreaCost', 'PostalCode', 'AvgLivingArea',..."
6,Gradient Boosting Regressor,False,MAPE,0.141873,6.0,0.10,200.0,"[AvgAreaCost, PostalCode, DistrictID, AvgLivin..."
7,Gradient Boosting Regressor,True,MAPE,0.009060,6.0,0.10,200.0,"[AvgAreaCost, PostalCode, DistrictID, AvgLivin..."
8,Gradient Boosting Regressor,False,MdAPE,0.093536,6.0,0.10,200.0,"[AvgAreaCost, PostalCode, DistrictID, AvgLivin..."
9,Gradient Boosting Regressor,True,MdAPE,0.006386,6.0,0.10,200.0,"[AvgAreaCost, PostalCode, DistrictID, AvgLivin..."


In [ ]:
save_csv(met_df, "metrics_summary.csv")